# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Structured Content Archetype Clustering**, so the ML task type is **clustering**. The unit is a content item, and the goal is to discover recurring, interpretable page profiles from observable structure such as content type, word count, content age, and update recency. This supports a content operations lead deciding which pages should be compared and reviewed together. There is no observed outcome target yet because this is an exploratory, unsupervised lane. A useful candidate evaluation is silhouette score plus a human interpretability check: clusters should be reasonably separated and understandable enough to guide editorial review.

In [10]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from io import BytesIO

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HimanshuSharma-2856/Flyrank-ml--internship/main/data/raw/content_refresh_anonymized.csv"
search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
relative_path = Path("data/raw/content_refresh_anonymized.csv")
candidates = [root / relative_path for root in search_roots]
local_path = next((path for path in candidates if path.exists()), None)

if local_path is not None:
    df = pd.read_csv(local_path)
    data_source = str(local_path)
else:
    last_error = None
    for attempt in range(1, 4):
        try:
            with urlopen(DATA_URL, timeout=120) as response:
                df = pd.read_csv(BytesIO(response.read()))
            data_source = DATA_URL
            break
        except (TimeoutError, URLError) as error:
            last_error = error
            print(f"Download attempt {attempt}/3 failed; retrying...")
    else:
        raise RuntimeError(
            "Could not download the starter CSV after 3 attempts. "
            "Check internet access or place the CSV at data/raw/content_refresh_anonymized.csv."
        ) from last_error

print(f"Data source: {data_source}")
print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns)}")

Data source: https://raw.githubusercontent.com/HimanshuSharma-2856/Flyrank-ml--internship/main/data/raw/content_refresh_anonymized.csv
Rows loaded: 30,000
Columns loaded: 44


## 2. Target or proxy

Clustering has **no supervised target**. Instead, the output is an `archetype_id` assigned after fitting a clustering method to the structural feature matrix. The candidate features are observed page properties, not future outcomes. I will not use `trend_direction`, `trend_pct`, or `is_declining_label` as clustering features because they are outcome/label-related and would change the question from descriptive archetype discovery into a different prediction task. The first target-like artifact is therefore a cluster assignment that must be checked for stability, sensible volume, and human interpretability.

In [11]:
label_related_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

available_label_related_columns = sorted(label_related_columns.intersection(df.columns))
print("Label-related columns excluded from clustering features:", available_label_related_columns)
print("Supervised target available for this lane: no; archetype_id will be created after clustering.")

Label-related columns excluded from clustering features: ['trend_direction', 'trend_pct']
Supervised target available for this lane: no; archetype_id will be created after clustering.


## 3. Success metric

The primary success check is **silhouette score**, interpreted alongside a human sense-check. A higher silhouette score indicates that items are more separated from neighboring clusters, but it does not prove that the archetypes are useful. I will call the result useful only when the clusters are also stable across reruns, have enough items to review, and can be described in plain language using the observed structural features. The downstream action is not to automate an editorial decision; it is to give the content operations lead a more coherent comparison set.

In [12]:
metric_name = "silhouette score plus human interpretability review"
print(f"Primary success metric: {metric_name}")
print("Good means: separated, stable, sufficiently sized, and interpretable archetypes that support editorial comparison.")

Primary success metric: silhouette score plus human interpretability review
Good means: separated, stable, sufficiently sized, and interpretable archetypes that support editorial comparison.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one row per pseudonymized content item/page**. I am using the full starter slice because the selected lane compares structural profiles across the content inventory. IDs are retained only for grouping and traceability, not as model features.

In [13]:
candidate_feature_columns = [
    "content_id",
    "client_id",
    "content_type",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

missing_columns = sorted(set(candidate_feature_columns) - set(df.columns))
assert not missing_columns, f"Missing expected columns: {missing_columns}"

lane_slice = df[candidate_feature_columns].copy()
lane_slice["archetype_id"] = pd.NA

assert len(lane_slice) == len(df)
assert lane_slice["content_id"].nunique() == len(lane_slice)
assert lane_slice["archetype_id"].isna().all()

print("Unit of analysis: one row per content item/page")
print(f"Rows in lane slice: {len(lane_slice):,}")
print("Lane slice preview:")
display(lane_slice.head(8))
print("Future output column:", "archetype_id (unassigned until clustering)")

Unit of analysis: one row per content item/page
Rows in lane slice: 30,000
Lane slice preview:


,content_id,client_id,content_type,word_count,content_age_days,days_since_last_update,archetype_id
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,187,20,<NA>
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,445,25,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,141,20,<NA>
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,463,22,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,263,14,<NA>
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,147,20,<NA>
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,90,20,<NA>
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,445,22,<NA>


Future output column: archetype_id (unassigned until clustering)


## 5. Why ML beats a fixed rule here

A fixed rule could split pages by one threshold, such as word count or content type, but that would miss combinations such as older pages with different structures, update recency, and missingness patterns. Clustering is worth testing because several observed signals may form recurring profiles that are difficult to specify honestly in advance. ML is not automatically better: if the clusters are unstable, uninterpretable, or no more useful than a simple content-type rule, I should keep the rule and report that finding. The action supported by a useful output is a prioritized comparison set for a content operations lead, not an automatic publish, delete, or refresh decision.

In [14]:
print("Fixed-rule baseline: compare pages by content_type alone.")
print("ML hypothesis: combinations of structure and recency may reveal useful, interpretable profiles beyond one threshold.")
print("Decision supported: help a content operations lead compare like with like and choose pages for review.")
print("Guardrail: retain the fixed rule if clustering is not stable, interpretable, or more useful.")

Fixed-rule baseline: compare pages by content_type alone.
ML hypothesis: combinations of structure and recency may reveal useful, interpretable profiles beyond one threshold.
Decision supported: help a content operations lead compare like with like and choose pages for review.
Guardrail: retain the fixed rule if clustering is not stable, interpretable, or more useful.


## Self-check

Before you submit, confirm each line honestly:

- [x] The task type is named: unsupervised clustering.
- [x] There is no invented supervised target; `archetype_id` is explicitly unassigned until clustering.
- [x] The success metric is named: silhouette score plus stability and human interpretability review.
- [x] The unit of analysis is shown as a real dataframe: one row per content item/page.
- [x] The output supports a real content action: comparison and editorial review by a content operations lead.
- [x] A fixed-rule baseline and a reason to test ML are stated.
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all).
- [x] No client names, private queries, or client-identifying URLs appear in the final notebook.
- [ ] The completed notebook is committed to the repository before submission.
